<a href="https://colab.research.google.com/github/AstonVSabrido/Final-Project-CS2/blob/main/Kamia_Transport_NB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import json
url = "https://raw.githubusercontent.com/AstonVSabrido/Final-Project-CS2/refs/heads/main/transport.json"
response = requests.get(url)
trips = response.json()
print(trips)


[{'trip_id': 101, 'route_id': 1, 'origin': 'Davao City', 'destination': 'Tagum', 'date': '2025-09-20', 'departure_time': '08:00', 'arrival_time': '09:30', 'travel_duration_min': 90, 'bus_operator': 'Davao Express', 'bus_type': 'Aircon', 'capacity': 45, 'passengers': 35, 'occupancy_rate': 77.8, 'fare': {'regular': 120, 'student': 96, 'senior': 84}, 'status': 'On Time'}, {'trip_id': 102, 'route_id': 1, 'origin': 'Davao City', 'destination': 'Tagum', 'date': '2025-09-20', 'departure_time': '14:00', 'arrival_time': '15:40', 'travel_duration_min': 100, 'bus_operator': 'Mindanao Transport', 'bus_type': 'Regular', 'capacity': 50, 'passengers': 48, 'occupancy_rate': 96.0, 'fare': {'regular': 110, 'student': 88, 'senior': 77}, 'status': 'Delayed'}, {'trip_id': 201, 'route_id': 2, 'origin': 'Davao City', 'destination': 'Mati', 'date': '2025-09-21', 'departure_time': '07:30', 'arrival_time': '11:00', 'travel_duration_min': 210, 'bus_operator': 'Davao Express', 'bus_type': 'Deluxe', 'capacity': 40

In [ ]:
!pip install firebase-admin

In [ ]:
import firebase_admin
from firebase_admin import credentials, db
# Load the private key
cred = credentials.Certificate("/content/firebase_key_project.json")
# Initialize the app with your database URL
firebase_admin.initialize_app(cred, {
 "databaseURL": "https://kamia-transport-db-default-rtdb.asia-southeast1.firebasedatabase.app/"
})
print("Firebase connected successfully!")

Firebase connected successfully!


In [ ]:
import requests
import json

with open('/content/transport.json', 'r') as f:
    data = json.load(f)

print("JSON file loaded")
ref = db.reference("trips")
for trip in data:
  ref.child(str(trip["trip_id"])).set(trip)
print("Data uploaded successfully!")

JSON file loaded
Data uploaded successfully!


In [24]:
ref = db.reference('trips')
trips_data = ref.get()

if trips_data:
    # Convert the dictionary of trips into a list of trip dictionaries
    trips_list = list(trips_data.values())
    print("Trips data retrieved successfully:")
    print(f"Number of trips: {len(trips_list)}")
    print("First trip:")
    print(trips_list[0])
else:
    trips_list = []
    print("No trips data found in Firebase.")

Trips data retrieved successfully:
Number of trips: 6
First trip:
{'arrival_time': '09:30', 'bus_operator': 'Davao Express', 'bus_type': 'Aircon', 'capacity': 45, 'date': '2025-09-20', 'departure_time': '08:00', 'destination': 'Tagum', 'fare': {'regular': 120, 'senior': 84, 'student': 96}, 'occupancy_rate': 77.8, 'origin': 'Davao City', 'passengers': 35, 'route_id': 1, 'status': 'On Time', 'travel_duration_min': 90, 'trip_id': 101}


In [27]:
global trips_data
global trips_list4

search_history = []

while True:
    print("\n--- Trip Filtering Application ---")
    print("1. Search for trips")
    print("2. View search history")
    print("3. Display all trips")
    print("4. Add a new trip")
    print("5. Exit")

    choice = input("Enter your choice: ")

    if choice == '1':
        print("Searching...")
        valid_criteria = ['destination', 'bus_type', 'trip_id', 'status']
        while True:
            search_criterion_input = input(f"Enter search criterion ({', '.join(valid_criteria)}): ").lower()
            if search_criterion_input in valid_criteria:
                search_criterion = search_criterion_input
                print(f"Selected search criterion: {search_criterion}")
                break
            else:
                print("Invalid criterion. Please choose from the available options.")

        # Extract unique values for the selected criterion
        # All string criteria should use their original case for unique values
        if search_criterion != 'trip_id':
            unique_values = sorted(list(set([str(trip[search_criterion]) for trip in trips_list if search_criterion in trip])))
        else:
            unique_values = sorted(list(set([trip[search_criterion] for trip in trips_list if search_criterion in trip])))
        print(f"\nAvailable {search_criterion} values: {', '.join(map(str, unique_values))}")

        # Prompt user for search value
        search_value_input = input(f"Enter the {search_criterion} you are looking for: ")

        # Convert search_value if criterion is trip_id, otherwise keep original case for string comparison
        if search_criterion == 'trip_id':
            try:
                search_value = int(search_value_input)
            except ValueError:
                print("Invalid input for Trip ID. Please enter a number.")
                continue
        else:
            search_value = search_value_input

        # Filter trips
        filtered_trips = []
        for trip in trips_list:
            if search_criterion in trip:
                trip_value = trip[search_criterion]
                if search_criterion != 'trip_id':
                    if str(trip_value) == search_value:
                        filtered_trips.append(trip)
                else:
                    # For trip_id, compare as integers
                    if trip_value == search_value:
                        filtered_trips.append(trip)

        # Display filtered trips
        if filtered_trips:
            print(f"\nFound {len(filtered_trips)} trip(s) matching '{search_value_input}' for {search_criterion}:")
            for trip in filtered_trips:
                search_history.append(trip) # Add trip to search history
                print(json.dumps(trip, indent=2))
        else:
            print(f"\nNo trips found matching '{search_value_input}' for {search_criterion}.")


    elif choice == '2':
        print("Viewing history...")
        if not search_history:
            print("No searches have been performed yet.")
        else:
            print("\n--- Search History ---")
            for i, trip in enumerate(search_history):
                print(f"\n--- Search History Entry {i + 1} ---")
                print(json.dumps(trip, indent=2))

    elif choice == "3":
        print("\n--- Trip List ---")
        if trips_list:
            for i, trip in enumerate(trips_list):
                print(f"\n--- Trip {i + 1} ---")
                print(json.dumps(trip, indent=2))
        else:
            print("No trips available.")


    elif choice == "4":
        print("\n--- Add a New Trip ---")
        try:
            new_trip = {}
            new_trip['trip_id'] = int(input("Enter Trip ID (integer): "))
            new_trip['route_id'] = int(input("Enter Route ID (integer): "))
            new_trip['origin'] = input("Enter Origin: ")
            new_trip['destination'] = input("Enter Destination: ")
            new_trip['date'] = input("Enter Date (YYYY-MM-DD): ")
            new_trip['departure_time'] = input("Enter Departure Time (HH:MM): ")
            new_trip['arrival_time'] = input("Enter Arrival Time (HH:MM): ")
            new_trip['travel_duration_min'] = int(input("Enter Travel Duration in minutes (integer): "))
            new_trip['bus_operator'] = input("Enter Bus Operator: ")
            new_trip['bus_type'] = input("Enter Bus Type: ")
            new_trip['capacity'] = int(input("Enter Capacity (integer): "))
            new_trip['passengers'] = int(input("Enter Number of Passengers (integer): "))

            if new_trip['capacity'] > 0:
                new_trip['occupancy_rate'] = round((new_trip['passengers'] / new_trip['capacity']) * 100, 1)
            else:
                new_trip['occupancy_rate'] = 0.0

            new_trip['fare'] = {}
            new_trip['fare']['regular'] = float(input("Enter Regular Fare: "))
            new_trip['fare']['student'] = float(input("Enter Student Fare: "))
            new_trip['fare']['senior'] = float(input("Enter Senior Fare: "))

            new_trip['status'] = input("Enter Status (e.g., On Time, Delayed, Cancelled): ")

            if str(new_trip['trip_id']) in trips_data:
                print(f"Trip with ID {new_trip['trip_id']} already exists. Please choose a different ID or modify an existing trip.")
            else:
                ref = db.reference("trips")
                ref.child(str(new_trip['trip_id'])).set(new_trip)
                print("Trip added successfully!")

                # Refresh trips_list and trips_data after adding a new trip
                trips_data_updated = ref.get()
                if trips_data_updated:
                    trips_list = list(trips_data_updated.values())
                    trips_data = trips_data_updated
                else:
                    trips_list = []
                    trips_data = {}

        except ValueError:
            print("Invalid input. Please ensure numeric fields are entered correctly.")
        except Exception as e:
            print(f"An error occurred: {e}")

    elif choice == '5':
        print("Exiting the application. Goodbye!")
        break
    else:
        print("Invalid choice. Please enter 1, 2, 3, 4, or 5.")


--- Trip Filtering Application ---
1. Search for trips
2. View search history
3. Display all trips
4. Add a new trip
5. Exit
Enter your choice: 1
Searching...
Enter search criterion (destination, bus_type, trip_id, status): destination
Selected search criterion: destination

Available destination values: Compostela, Mati, Tagum
Enter the destination you are looking for: Tagum

Found 2 trip(s) matching 'Tagum' for destination:
{
  "arrival_time": "09:30",
  "bus_operator": "Davao Express",
  "bus_type": "Aircon",
  "capacity": 45,
  "date": "2025-09-20",
  "departure_time": "08:00",
  "destination": "Tagum",
  "fare": {
    "regular": 120,
    "senior": 84,
    "student": 96
  },
  "occupancy_rate": 77.8,
  "origin": "Davao City",
  "passengers": 35,
  "route_id": 1,
  "status": "On Time",
  "travel_duration_min": 90,
  "trip_id": 101
}
{
  "arrival_time": "15:40",
  "bus_operator": "Mindanao Transport",
  "bus_type": "Regular",
  "capacity": 50,
  "date": "2025-09-20",
  "departure_ti